# Data Preprocess 

This script concatenates most recent passive data with backup data and performs basic preprocessing on passive, EMA and Monitoring data.

In [93]:
from pyprojroot import here # define relative paths to the project root (working directory)
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc  
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------

# relative project root
PROJECT_ROOT = here() # '.here' is located as invisible file in the project root working directory
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path, preprocessed_path_freezed
from missing_data import compute_availability_metrics
from functions.preprocessing.infer_timeoffset import (
    create_utcday_tzoffset_df,
    merge_fill_tz,
)

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")        
today_day = pd.Timestamp.today().normalize()       

today_str = "23022026"
# --- Path -------------------------------------------------------------

datapath = Path(raw_path) / f"export_tiki_{today_str}"  

## 1. Passive Data

### 1.1 Load most recent passive data

In [2]:
# actual passive + ema_data

# define the pattern for passive data files
file_pattern = os.path.join(datapath, "epoch_part*.csv")

# use glob to find all matching files
file_list = glob.glob(file_pattern)

# sort the file list for consistent ordering
file_list.sort()

# concatenate all CSV files into a single DataFrame
df_complete = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)


In [3]:
# Extract customer identifier and reduce to first 4 characters
df_complete["customer"] = df_complete.customer.str.split("@").str.get(0)
df_complete["customer"] = df_complete["customer"].str[:4]

for col in ["startTimestamp", "endTimestamp"]:
    df_complete[col] = (
        pd.to_datetime(df_complete[col], utc=True, errors="coerce", unit="ms")
    )

#df_complete

In [4]:
df_complete = df_complete[['customer', 'startTimestamp', 'endTimestamp', 'timezoneOffset', 'type',
       'stringValue', 'booleanValue', 'doubleValue', 'longValue']]

### 1.2 Load big backup dataset

In [5]:
# merge with backup data
backup_path = preprocessed_path + "/backup_passive_05052025.parquet"

# backup_path = preprocessed_path + "/backup_passive_last.feather"
df_backup = pd.read_parquet(backup_path)

In [6]:
# make independent copies of both DataFrames to avoid SettingWithCopyWarning (future modifications do not affect any original DataFrame)
df_backup = df_backup.copy()
df_complete = df_complete.copy()

# convert booleanValue to boolean[pyarrow] dtype before concatenation so that it can be saved to .feather later
# alternative to "boolean[pyarrow]" is "boolean", but it is experimental and may change in future pandas versions
df_backup['booleanValue'] = df_backup['booleanValue'].astype('boolean[pyarrow]')
df_complete['booleanValue'] = df_complete['booleanValue'].astype('boolean[pyarrow]')

In [7]:
# latest timestamp from the backup dataset
latest_timestamp = df_backup['startTimestamp'].max()

# filter from df_complete only those entries that are newer than what’s already in the backup
df_complete_filtered = df_complete[df_complete['startTimestamp'] > latest_timestamp]

### 1.3 Concat Backup and most recent data

In [8]:
# update the backup by concatenating only the newly filtered rows from df_complete, creating an up-to-date backup
df_backup_recent = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

### 1.4 Rename variable names and create additional columns 

In [9]:
# define a clear mapping for rename columns
rename_columns = {"customer": "id",
                  "type": "modality",
                  "startTimestamp": "timestamp_start",
                  "endTimestamp": "timestamp_end",
                  "booleanValue": "boolean_value",
                  "doubleValue": "double_value",
                  "longValue": "long_value",
                  "timezoneOffset": "timezone_offset"}

# apply renaming
df_backup_recent = df_backup_recent.rename(columns=rename_columns)

In [10]:
# create a unified float_value column:
# use 'doubleValue' where available (more precise), otherwise use 'longValue'
df_backup_recent['float_value'] = df_backup_recent['double_value'].fillna(df_backup_recent['long_value'])


In [11]:
# drop original value columns that have been unified into 'float_value' + 'stringValue' (because only ECG data are stored as string for period March - November 2023) + 'createdtAt'
df_backup_recent = df_backup_recent.drop(columns=['double_value', 'long_value', 'stringValue'])


In [12]:
# create a time_interval (duration in seconds) column
df_backup_recent['time_interval'] = (
    df_backup_recent['timestamp_end'] - df_backup_recent['timestamp_start']
).dt.total_seconds()

# create a start_date and start_hour column
df_backup_recent['start_date']  = df_backup_recent['timestamp_start'].dt.normalize()
df_backup_recent['start_hour'] = df_backup_recent['timestamp_start'].dt.hour

What's next?

1. Since we want to include 'for_id' and 'study_version' in our `passive_data` data frame, we need to extract these data from the monitoring sheet. This is done in section 2

2. Additionally the `monitoring_data` data frame is set up in section 2


### 1.5 Infer Timezone Offset

In [13]:
df_tz = create_utcday_tzoffset_df(
    df_backup_recent,
    customer_col="id",
    startTimestamp_col="timestamp_start",
    timezoneOffset_col="timezone_offset",
    type_col="modality",
    createdAt_col="createdAt",
)


2026-02-27 15:49:21,701 [INFO] [GPS] Could not infer tz for customer NGlo at 2023-11-06 00:00:00+00:00 with potential tzs [  60. -300.    0.] - gps_multiple_conflict_with_previous_no_next
2026-02-27 15:49:25,106 [INFO] [ActivityDetailCreatedAt] Could not infer tz for customer S5sH at 2024-07-16 00:00:00+00:00 with potential tzs [420. 180.] - activitydetail_multiple_conflict_with_both
2026-02-27 15:49:56,235 [INFO] [Interpolate] Could not infer tz for customer 3oNs at 2024-07-08 00:00:00+00:00 - interpolate_no_previous_no_next
2026-02-27 15:49:56,255 [INFO] [Interpolate] Could not infer tz for customer 3oNs at 2024-07-09 00:00:00+00:00 - interpolate_no_previous_no_next
2026-02-27 15:51:18,574 [INFO] [Interpolate] Could not infer tz for customer 9nSQ at 2024-04-12 00:00:00+00:00 - interpolate_no_previous_no_next
2026-02-27 15:51:33,659 [INFO] [Interpolate] Could not infer tz for customer AY8z at 2025-06-10 00:00:00+00:00 - interpolate_no_previous_no_next
2026-02-27 15:51:33,678 [INFO] [I

In [14]:
df_tz.head()

,id,day,inferred_tzoffset,inferred_tzoffset_source
0,05kz,2023-10-10 00:00:00+00:00,120.0,gps_single
1,05kz,2023-10-11 00:00:00+00:00,120.0,gps_single
2,05kz,2023-10-12 00:00:00+00:00,120.0,gps_single
3,05kz,2023-10-13 00:00:00+00:00,120.0,gps_single
4,05kz,2023-10-14 00:00:00+00:00,120.0,gps_single


In [15]:
df_backup_recent = df_backup_recent.merge(
    df_tz,
    left_on=["id", "start_date"],
    right_on=["id", "day"],
    how="left",
)
#df_backup_recent.drop(columns=["day"], inplace=True)  # remove day from df_tz

In [16]:
assert df_backup_recent.inferred_tzoffset.isna().sum() == 0, (
    "There are missing inferred timezone offsets!"
)

In [18]:
# just to make sure that we don't use them anymore later
del df_complete_filtered
del df_complete

## 2. Monitoring data

In [19]:
# import data
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [20]:
# get an overview of the monitoring data
#df_monitoring.head()

In [21]:
# show columns of monitoring data
print(df_monitoring.columns.tolist())

['FOR_ID', 'EMA_ID', 'Pseudonym', 'Studienversion', 'Status', 'Besonderes', 'T20=Post', 'Start EMA Baseline', 'Ende EMA Baseline', 'Terminpräferenz', 'Termin 1. Gespräch', 'Wer führt 1. Gespräch?', 'Telefonat stattgefunden?', 'Baseline T20 Update verschickt?', 'Freischaltung/ Start EMA T20', 'Ende EMA T20', 'Freischaltung/ Start EMA Post', 'Ende EMA Post', 'Post T20 Update verschickt?', 'Studienende/ Dropout Mail verschickt?', 'Status reduziert', 'Dropout vor KV', 'Dropout nach KV', 'Dropout Grund (frei)', 'Zeitpunkt SP6 Dropout (last day passive/gps)']


In [ ]:
df_monitoring = df_monitoring.copy()

df_monitoring.rename(columns = {"Pseudonym": "id",
                                "FOR_ID": "for_id",
                                "EMA_ID": "ema_id", 
                                "Status": "study_status",
                                "Studienversion":"study_version", 
                                "Start EMA Baseline": "ema_base_start", 
                                "Ende EMA Baseline": "ema_base_end", 
                                "Freischaltung/ Start EMA T20": "ema_t20_start",
                                "Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", 
                                "T20=Post":"t20_post" }, 
                     inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'id', 'study_version', 'study_status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end']]

df_monitoring["id"] = df_monitoring["id"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)


### 2.1 Merge relevant columns with passive data

In [23]:
df_monitoring_short = df_monitoring[["id", "for_id","study_version"]]

#### 2.2 Final `passive_data` data frame

In [24]:
df_backup_recent= df_backup_recent.merge(df_monitoring_short, on="id", how="right")

In [25]:
df_backup_recent.head()

,id,timestamp_start,timestamp_end,timezone_offset,modality,boolean_value,createdAt,float_value,time_interval,start_date,start_hour,day,inferred_tzoffset,inferred_tzoffset_source,for_id,study_version
0,4MLe,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,120.0,Steps,<NA>,NaT,6.00,60.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,FOR11905,Lang
1,4MLe,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,120.0,ActiveBurnedCalories,<NA>,NaT,0.14,60.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,FOR11905,Lang
2,4MLe,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,120.0,CoveredDistance,<NA>,NaT,4.62,60.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,FOR11905,Lang
3,4MLe,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,120.0,HeartRate,<NA>,NaT,74.00,37.0,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,FOR11905,Lang
4,4MLe,2023-05-17 18:00:55+00:00,NaT,120.0,HeartRate,<NA>,NaT,95.00,NaN,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,60.0,interpolate_no_previous_next_is_fine,FOR11905,Lang


In [26]:
# ensure data types are coded correctly
#df_backup_recent['boolean_value'] = df_backup_recent['boolean_value'].astype('boolean[pyarrow]')
#df_backup_recent['study_version'] = df_backup_recent['study_version'].astype('string')
#df_backup_recent['modality'] = df_backup_recent['modality'].astype('string')

In [27]:
# Get a list of columns to drop (all columns not in keep_cols)
keep_cols_passive = ['id','for_id', 'modality', 'timestamp_start','timestamp_end','time_interval', 'float_value', 'boolean_value','start_date', 
    'start_hour', "timezone_offset", 'study_version']

# final passive data frame
df_passive_final = df_backup_recent[keep_cols_passive]

#### 2.3 Final `monitoring_data` data frame

In [28]:
#??

## 3. EMA data

#### 3.1 Load, match and rename relevant data from separate .csv files

In [29]:

# Beispiel: datapath = Path("/pfad/zum/verzeichnis")
session        = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers        = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice         = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions      = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire  = pd.read_csv(datapath / "questionnaires.csv", low_memory=False)  # ohne Komma!


In [30]:
# session data
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 
session["user"] = session["user"].str[:4]
session.rename(columns = {"id":"session_id",
                          "user":"id",
                          "completedAt": "timestamp_beep_completion", 
                          "createdAt": "timestamp_item_completion", 
                          "expirationTimestamp": "timestamp_beep_expiration",
                          "sessionRun":"beep_num_run",
                          "study":"schedule_chronotype"}, inplace=True)
session['measurement_burst'] = session['schedule_chronotype'].map(study_mapping)
session['schedule_chronotype'] = session['schedule_chronotype'].map(chronotype_mapping)
# Parse epoch‑ms columns as UTC and drop tz info -> naive UTC
for col in ["timestamp_item_completion", "timestamp_beep_completion", "timestamp_beep_expiration"]:
    session[col] = (
        pd.to_datetime(session[col], unit="ms", utc=True, errors="coerce")
    )

df_sess = session[["id","session_id", "measurement_burst", "beep_num_run", "timestamp_item_completion", "timestamp_beep_completion", "schedule_chronotype", "timestamp_beep_expiration"]]

In [31]:
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
chronotype_mapping = {24: 0, 25: 1, 33: 0, 34: 1, 38: 0, 39: 1} 

answers["user"] = answers["user"].str[:4]
answers = answers[["user", "questionnaire", "study", "question", "element",
                   "createdAt", "order", "questionnaireSession"]]

answers["createdAt"] = (
    pd.to_datetime(answers["createdAt"], unit="ms", utc=True, errors="coerce")
)
answers['measurement_burst'] = answers['study'].map(study_mapping)
answers['schedule_chronotype'] = answers['study'].map(chronotype_mapping)

answers.rename(columns={"user": "id", 
                        "createdAt": "timestamp_item_completion",
                        "questionnaire": "beep_type",
                        "question":"item_code_map",
                        "order":"item_order",
                        "questionnaireSession":"session_id",
                        }, inplace=True)
answers.drop(columns=["study"], inplace=True)

In [32]:
# item description data
choice = choice[["element", "choice_id", "text", "question"]]
choice.rename(columns={"text":"response_text",
                       "choice_id":"response", 
                       "question":"item_code_map"}, inplace=True)

In [33]:
# question description data
questions = questions[["id", "title"]]
questions.rename(columns={"id":"item_code_map",
                          "title":"item"}, inplace=True)

In [34]:
questionnaire = questionnaire[["id", "name"]]
questionnaire.rename(columns={"id":"beep_type",
                              "name":"beep_type_name"}, inplace=True)

In [35]:
answer_merged = pd.merge(answers, choice, on= ["item_code_map","element"])
answer_merged = pd.merge(answer_merged, questions, on= "item_code_map")
answer_merged = pd.merge(answer_merged, questionnaire, on= "beep_type")
answer_merged["date"] = answer_merged["timestamp_item_completion"].dt.normalize()

#### 3.2 Calculate auxiliary variables

In [36]:
df_monitoring_ema = df_monitoring[["id", "for_id","study_version", 'ema_base_start', 'ema_base_end',
       'ema_t20_start', 'ema_t20_end', 'ema_post_start', 'ema_post_end', 't20_post']]

In [37]:
bursts = [("base", 0), ("t20", 1), ("post", 2)]

out = []
for burst_name, burst_code in bursts:
    tmp = df_monitoring_ema[[
        "id", "for_id", "study_version", "t20_post",
        f"ema_{burst_name}_start", f"ema_{burst_name}_end"
    ]].copy()

    tmp = tmp.rename(columns={
        f"ema_{burst_name}_start": "ema_burst_start",
        f"ema_{burst_name}_end":   "ema_burst_end",
    })
    tmp["measurement_burst"] = burst_code
    out.append(tmp)

df_monitoring_ema_long = (
    pd.concat(out, ignore_index=True)
      # optional: drop rows where the burst is entirely missing
      .dropna(subset=["ema_burst_start", "ema_burst_end"], how="all")
      .sort_values(["id", "measurement_burst"])
      .reset_index(drop=True)
)


In [39]:
answer_merged = pd.merge(answer_merged, df_monitoring_ema_long, on = ["id", "measurement_burst"])

In [40]:
df_ema_content = answer_merged.copy()

In [41]:
# 1. Date and Time Manipulations
df_ema_content['weekday'] = df_ema_content['timestamp_item_completion'].dt.day_name()
df_ema_content['date'] = df_ema_content['timestamp_item_completion'].dt.floor('D')

# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    elif month in [9, 10, 11]:
        return 3
    else:
        return 4

df_ema_content['season'] = df_ema_content['timestamp_item_completion'].dt.month.apply(get_season)

# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return 1
    elif 8 <= hour < 12:
        return 2
    elif 12 <= hour < 17:
        return 3
    elif 17 <= hour < 21:
        return 4
    else:
        return 5

df_ema_content['time_of_day'] = df_ema_content['timestamp_item_completion'].dt.hour.apply(get_time_of_day)

# 2. Study Mapping / String Manipulation
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
df_ema_content['item'] = df_ema_content['timestamp_item_completion'].dt.strftime('%Y-%m-%d') + '_' + df_ema_content['item'].str.replace('_morning', '', regex=False)

# 3. Weekend Indicator
df_ema_content['weekend'] = df_ema_content['weekday'].isin(['Saturday', 'Sunday']).astype(int)

# 4. Questionnaire Number
df_ema_content['quest_nr'] = df_ema_content['beep_type_name'].str.extract(r'(\d+)').astype(float)

# 5. Count unique questionnaires per day
df_ema_content['n_quest'] = (
    df_ema_content.groupby(['measurement_burst','schedule_chronotype', 'id', 'date'])['beep_type_name']
                  .transform('nunique')
)

# 6. Unique Day Identifier
df_ema_content['quest_nr_str'] = df_ema_content['n_quest'].fillna('unknown').astype(str)
df_ema_content['beep_per_person_id'] = (
    df_ema_content['date'].dt.strftime('%Y%m%d') + '_' + df_ema_content['quest_nr_str']
)
df_ema_content.drop(columns=['quest_nr_str'], inplace=True)  # remove intermediate column

# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content['ema_relat_burst_start'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('min')
)
df_ema_content['ema_relat_burst_end'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('max')
)

# 8. Absolute & Relative Day Index
df_ema_content['absolute_day_index'] = (
    df_ema_content['date'] - df_ema_content['ema_relat_burst_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date']
                  .rank(method='dense').astype(int)
)

# 9. Filter absolute_day_index > 16
max_allowed_days = 16
df_ema_content = df_ema_content[df_ema_content['absolute_day_index'] <= max_allowed_days].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['id'].unique())
else:
    print("All entries have absolute_day_index <= 16.")

# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(subset=['id', 'measurement_burst', 'beep_per_person_id']).copy()
df_unique['relative_beep_counter'] = df_unique.groupby(['id', 'measurement_burst']).cumcount() + 1
df_ema_content = df_ema_content.merge(
    df_unique[['id', 'measurement_burst', 'beep_per_person_id', 'relative_beep_counter']],
    on=['id', 'measurement_burst', 'beep_per_person_id'],
    how='left'
)

# 12. Missing Data behandeln
df_ema_content['measurement_burst'] = df_ema_content['measurement_burst'].fillna('unknown')
df_ema_content['absolute_day_index'] = df_ema_content['absolute_day_index'].where(
    df_ema_content['ema_relat_burst_start'].notna(), np.nan
)



All entries have absolute_day_index <= 16.


In [43]:
meta_cols = ['id','for_id','response_text','item_code_map','beep_type' ,'beep_type_name',
              'element', 'item_order', 'session_id']
df_ema_meta = df_ema_content[meta_cols].copy()

In [44]:
df_sess_short = df_sess[["id", "session_id",  "beep_num_run",'timestamp_beep_completion']].copy()

In [45]:
df_ema_meta = df_ema_meta.merge(df_sess_short, on=["id", "session_id"], how="left")

#### 3.3 Calculate EMA coverage

In [56]:
df_sess_short_compliance = df_sess[["id", "session_id", 'timestamp_beep_completion']].copy()

In [ ]:
df_ema_content = df_ema_content.merge(df_sess_short_compliance, on=["id", "session_id"], how="left")

In [67]:
df_ema_content["n_beeps_beeps_completed_per_burst"] = (
    df_ema_content
    .groupby(["measurement_burst", "id"])["timestamp_beep_completion"]
    .transform("nunique"))

#### 3.3 Calculate auxiliary variables

In [72]:
df_ema_content = answer_merged.copy()

In [76]:
# 1. Date and Time Manipulations
df_ema_content['weekday'] = df_ema_content['timestamp_item_completion'].dt.day_name()
df_ema_content['createdAt_day'] = df_ema_content['timestamp_item_completion'].dt.floor('D')


# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'Winter'

df_ema_content['season'] = df_ema_content['timestamp_item_completion'].dt.month.apply(get_season)

# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return 'Early Morning'
    elif 8 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df_ema_content['time_of_day'] = df_ema_content['timestamp_item_completion'].dt.hour.apply(get_time_of_day)
df_ema_content['item'] = df_ema_content['item'].str.replace('_morning', '', regex=False)

# 3. Weekend Indicator
df_ema_content['weekend'] = df_ema_content['weekday'].isin(['Saturday', 'Sunday']).astype(int)

# 4. Questionnaire Number
df_ema_content['nr_beep_daily'] = df_ema_content['beep_type_name'].str.extract(r'(\d+)').astype(float)

# 5. Count unique questionnaires per day
df_ema_content['n_beeps_completed_per_day'] = (
    df_ema_content.groupby(['measurement_burst', 'id', 'date'])['beep_type_name']
                  .transform('nunique')
)

# 6. Unique Day Identifier
df_ema_content['quest_nr_str'] = df_ema_content['nr_beep_daily'].fillna('unknown').astype(str)
df_ema_content['beep_per_person_id'] = (
    df_ema_content['date'].dt.strftime('%Y%m%d') + '_' + df_ema_content['quest_nr_str']
)

# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content['ema_relative_start'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('min')
)
df_ema_content['ema_relative_end'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date'].transform('max')
)

# 8. Absolute & Relative Day Index
df_ema_content['absolute_day_index'] = (
    df_ema_content['date'] - df_ema_content['ema_relative_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['id', 'measurement_burst'])['date']
                  .rank(method='dense').astype(int)
)

# 9. Filter absolute_day_index > 16
max_allowed_days = 16
df_ema_content = df_ema_content[df_ema_content['absolute_day_index'] <= max_allowed_days].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['customer'].unique())
else:
    print("All entries have absolute_day_index <= 16.")

# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(subset=['id', 'measurement_burst', 'beep_per_person_id']).copy()
df_unique['relative_beep_counter'] = df_unique.groupby(['id', 'measurement_burst']).cumcount() + 1
df_ema_content = df_ema_content.merge(
    df_unique[['id', 'measurement_burst', 'beep_per_person_id', 'relative_beep_counter']],
    on=['id', 'measurement_burst', 'beep_per_person_id'],
    how='left'
)

# 12. Missing Data behandeln
df_ema_content['measurement_burst'] = df_ema_content['measurement_burst'].fillna('unknown')
df_ema_content['absolute_day_index'] = df_ema_content['absolute_day_index'].where(
    df_ema_content['ema_relative_start'].notna(), np.nan
)

All entries have absolute_day_index <= 16.


In [ ]:
df_complete_ema_final = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

In [ ]:
df_complete_ema_final['id'] = df_complete_ema_final['id'].astype('category')
df_complete_ema_final['type'] = df_complete_ema_final['type'].astype('category')

# Downcast numeric columns
df_complete_ema_final['doubleValue'] = pd.to_numeric(df_complete_ema_final['doubleValue'], errors='coerce', downcast='float')
df_complete_ema_final['longValue'] = pd.to_numeric(df_complete_ema_final['longValue'], errors='coerce', downcast='integer')

### 3.4 merge the inferred tz offsets

In [94]:
# uncomment if want to run this cell multiple times
# if "inferred_tzoffset" in df_sess.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_sess")
#     df_sess = df_sess.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])
# if "inferred_tzoffset" in df_ema_content.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_ema_content")
#     df_ema_content = df_ema_content.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])

df_ema_meta = merge_fill_tz(
    df_ema_meta, df_tz, day_col="date", customer_col="id"
)
df_ema_content = merge_fill_tz(
    df_ema_content, df_tz, day_col="date", customer_col="id"
)

KeyError: 'customer'

### Export passive, EMA and Monitoring

In [ ]:
backup_path = raw_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(backup_path)

preprocessed_path_final = preprocessed_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(preprocessed_path_final)

#preprocessed_path_freezed_final = preprocessed_path_freezed + "/code_check" + "/backup_passive_recent.parquet"
#df_passive_final.to_parquet(preprocessed_path_freezed_final)

with open(preprocessed_path + '/ema_adherence_data.pkl', 'wb') as file:
    pickle.dump(df_sess, file)

    
with open(preprocessed_path + '/monitoring_data.pkl', 'wb') as file:
    pickle.dump(df_monitoring, file)

    
with open(preprocessed_path + '/ema_content.pkl', 'wb') as file:
    pickle.dump(df_ema_content, file)

In [ ]:

# Export df_adherence as CSV
df_sess_csv_path = preprocessed_path + '/ema_adherence_data.csv'
df_sess.to_csv(df_sess_csv_path, index=False)

# Export df_monitoring as CSV
df_monitoring_csv_path = preprocessed_path + '/monitoring_data.csv'
df_monitoring.to_csv(df_monitoring_csv_path, index=False)

# Export df_ema_content as CSV
df_ema_content_csv_path = preprocessed_path + '/ema_content.csv'
df_ema_content.to_csv(df_ema_content_csv_path, index=False)

# Export df_ema_content as CSV to freezed for data check
#df_ema_content_csv_path = preprocessed_path_freezed +'/code_check' +'/ema_content_recent.csv'
#df_ema_content.to_csv(df_ema_content_csv_path, index=False)

